### RAG Tokenization Demo

In a Retrieval-Augmented Generation (RAG) system, we process raw text by breaking it down into smaller units called **tokens** (such as words or subwords). These tokens are then converted into numerical representations (embeddings) so that a retriever can search for relevant documents in a vector database.

In [8]:
# Define a sample sentence for our RAG concept
sentence = "Retrieval-Augmented Generation improves LLM responses using external data."

# Perform a basic whitespace and punctuation split to create token strings
import re
tokens = re.findall(r'\w+|[^\w\s]', sentence)

print(f"Original Sentence: {sentence}\n")
print(f"Generated Tokens:  {tokens}\n")
print(f"Total Number of Tokens: {len(tokens)}")

Original Sentence: Retrieval-Augmented Generation improves LLM responses using external data.

Generated Tokens:  ['Retrieval', '-', 'Augmented', 'Generation', 'improves', 'LLM', 'responses', 'using', 'external', 'data', '.']

Total Number of Tokens: 11


In [2]:
import pandas as pd

In [4]:
df =pd.read_csv('/content/gender_submission.csv')

In [5]:
df

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


### Converting Tokens to Embeddings for RAG

To use these tokens or sentences in a retrieval system, we must convert them into numerical vectors called **embeddings**.

We will install `sentence-transformers` to load a pre-trained model and generate these embeddings.

In [11]:
!pip install -q sentence-transformers

In [13]:
from sentence_transformers import SentenceTransformer

# Initialize a lightweight, popular model for generating text embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for each of our individual tokens
token_embeddings = model.encode(tokens)

# Also generate an embedding for the entire sentence (which is what we typically store in a RAG vector DB)
sentence_embedding = model.encode(sentence)

print(f"Shape of each token embedding: {token_embeddings[0].shape}")
print(f"Shape of the overall sentence embedding: {sentence_embedding.shape}\n")

# Preview the embedding vector for the first token
print(f"Embedding vector for token '{tokens[0]}':\n{token_embeddings[0][:10]} ... (truncated)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape of each token embedding: (384,)
Shape of the overall sentence embedding: (384,)

Embedding vector for token 'Retrieval':
[-0.02429283  0.03640135 -0.03477852  0.0058078  -0.02285974  0.0890498
  0.12123766 -0.00662381  0.01191258 -0.0672664 ] ... (truncated)


### Extracting and Chunking Text from PDF for RAG

To build a RAG system using your `/content/college-rules.pdf` document, we need to:
1. **Extract** the raw text from the PDF pages.
2. **Chunk** the extracted text into smaller, overlapping segments so that we retain context while fitting the token limits of our embedding model.

In [17]:
!pip install -q pypdf

In [18]:
from pypdf import PdfReader

# Load the PDF file
pdf_path = "/content/college-rules.pdf"
reader = PdfReader(pdf_path)

# Extract text from all pages
full_text = ""
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        full_text += f"\n--- Page {i+1} ---\n" + text

print(f"Successfully extracted text from {len(reader.pages)} pages.")
print(f"Total length of extracted text: {len(full_text)} characters.\n")

# Preview the first 500 characters of the extracted text
print("--- Sample Extracted Text ---")
print(full_text[:500] + "...")

Successfully extracted text from 6 pages.
Total length of extracted text: 15322 characters.

--- Sample Extracted Text ---

--- Page 1 ---
1. General Rules and Regulations  
1.1  The Principal of the College shall be the disciplinary authority in respect of the students 
in the college. 
1.2  Ragging is strictly banned on the college campus. Anyone found guilty of     ragging 
and/or abetting ragging will be punished as per the directives of the UGC, Government 
and the Supreme Court. All the incidents of ragging will be severely dealt with. Those 
found guilty will be dismissed from the college and police cases wil...


Now that we have the text, we can implement a basic sliding-window chunking function. Creating overlapping chunks (e.g., chunk size of 500 characters with an overlap of 100 characters) ensures that semantic meaning isn't lost across chunk boundaries.

In [19]:
def chunk_text(text, chunk_size=500, chunk_overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - chunk_overlap)
    return chunks

# Chunk the extracted PDF text
pdf_chunks = chunk_text(full_text, chunk_size=600, chunk_overlap=150)
print(f"Created {len(pdf_chunks)} chunks from the PDF.")

# Preview the first chunk
print("\n--- First Chunk Sample ---")
print(pdf_chunks[0])

Created 35 chunks from the PDF.

--- First Chunk Sample ---

--- Page 1 ---
1. General Rules and Regulations  
1.1  The Principal of the College shall be the disciplinary authority in respect of the students 
in the college. 
1.2  Ragging is strictly banned on the college campus. Anyone found guilty of     ragging 
and/or abetting ragging will be punished as per the directives of the UGC, Government 
and the Supreme Court. All the incidents of ragging will be severely dealt with. Those 
found guilty will be dismissed from the college and police cases will be filed against 
them.  
1.3  In accordance with the policy adopted by the Governing Body and the
